# 10.01 云端 MindIE 推理服务

## 本节目标

- 下载 Qwen2-7B-Instruct 的固定版本模型快照。
- 使用一张 Ascend NPU 生成 MindIE 配置并启动单机推理服务。
- 通过 OpenAI 兼容接口完成一次对话请求，为 10.02 的端云切换页面提供云端后端。

## 实验原理

### 端云调用链路

模型权重和 MindIE 服务运行在云端。开发板上的 Gradio 页面切换到云端模式后，先访问开发板本机的 <code>127.0.0.1:1025</code>；SSH 本地转发将请求送到云端同一回环端口上的 MindIE 服务。

~~~text
浏览器
  │
  ▼
开发板 Gradio
  │  云端模式
  ▼
开发板 127.0.0.1:1025 ── SSH 本地转发 ──► 云端 127.0.0.1:1025
                                                   │
                                                   ▼
                                         MindIE + Qwen2-7B-Instruct
~~~

浏览器始终访问开发板上的聊天页面。实验结束时，用同一问题分别询问端侧和云端，记录首轮耗时、回答内容和一次连续追问。

### MindIE 单机服务

![MindIE Service 架构图](images/mindie-service-architecture.png)

图中 Endpoint 接收 RESTful 请求，BackendManager 管理推理后端，MindIE LLM 执行模型推理。本册使用 MindIE 3.0 的单机服务形态：配置文件写入模型权重路径、监听端口和 NPU 信息，服务进程加载权重后向客户端提供 HTTP 接口。

图：MindIE Service 架构图。来源：[华为昇腾 MindIE Service 产品简介](https://www.hiascend.com/document/detail/zh/mindie/100/mindieservice/servicedev/mindie_service0001.html)。

## 实验环境

云端运行目录以仓库目录名结尾：<code>&lt;云端工作目录&gt;/10_end-cloud_dual-mode_dialogue_system</code>。在启动 JupyterLab 的终端设置 <code>LAB10_CLOUD_ROOT</code> 后，模型、配置和日志会写入这个目录。

~~~bash
export LAB10_CLOUD_ROOT="/home/<云端用户名>;/work/10_end-cloud_dual-mode_dialogue_system"
jupyter lab
~~~

Kernel 是 Notebook 代码实际使用的 Python 进程。首次使用时，用任意可用 Kernel 运行下一节的注册单元。该单元在镜像已有的 <code>mindie-3.0.0</code> Conda 环境中安装 ModelScope，并注册 <code>lab10-cloud-mindie</code>。随后在 Jupyter 菜单中切换到 <code>Python (Lab10 Cloud MindIE)</code>，重启 Kernel，再从头执行本 Notebook。

课程镜像提供 CANN、NPU 驱动和 MindIE。MindIE 环境脚本会设置 Python 包、运行时库和 NPU 设备相关变量，服务进程在这套环境中启动。

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import shlex
import shutil
import signal
import socket
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

lab = {}

## 实验流程

### 1. 注册云端 Kernel

ModelScope 用于下载模型快照。本册固定使用 <code>modelscope==1.39.1</code>，并将 Qwen2-7B-Instruct 固定到一个 revision；同一份实验记录能够对应同一版本的模型文件。安装命令在 <code>mindie-3.0.0</code> Conda 环境中执行，并使用 PyPI。

注册完成后切换到 <code>Python (Lab10 Cloud MindIE)</code> 并重启 Kernel。后续单元会显示 Python 解释器路径，模型下载和服务启动均由该环境完成。

In [ ]:
ENV_ROOT = Path('/home/ma-user/anaconda3/envs/mindie-3.0.0')
ENV_PYTHON = ENV_ROOT / 'bin' / 'python'
KERNEL_NAME = 'lab10-cloud-mindie'
KERNEL_DISPLAY_NAME = 'Python (Lab10 Cloud MindIE)'
MODELSCOPE_PACKAGE = 'modelscope==1.39.1'
PYPI_INDEX_URL = 'https://pypi.org/simple'

if not ENV_PYTHON.is_file():
    raise FileNotFoundError(f'找不到 MindIE Python：{ENV_PYTHON}')

ipykernel_check = subprocess.run(
    [str(ENV_PYTHON), '-m', 'pip', 'show', 'ipykernel'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=False,
).returncode

packages = [MODELSCOPE_PACKAGE]
if ipykernel_check != 0:
    packages.append('ipykernel')

subprocess.run(
    [
        str(ENV_PYTHON), '-m', 'pip', 'install',
        '--disable-pip-version-check',
        '--index-url', PYPI_INDEX_URL,
        '--no-input',
        '--upgrade-strategy', 'only-if-needed',
        *packages,
    ],
    check=True,
)
subprocess.run(
    [
        str(ENV_PYTHON), '-m', 'ipykernel', 'install', '--user',
        '--name', KERNEL_NAME,
        '--display-name', KERNEL_DISPLAY_NAME,
    ],
    check=True,
)

print('ModelScope 安装源：', PYPI_INDEX_URL)
print('已注册 Kernel：', KERNEL_DISPLAY_NAME)
print('请切换到该 Kernel，重启后从本 Notebook 第一格重新运行。')

### 2. 准备运行目录与环境

切换 Kernel 后运行本节。所有运行产物位于 Lab 10 目录：

~~~text
10_end-cloud_dual-mode_dialogue_system/
├── models/Qwen2-7B-Instruct/
├── cache/modelscope/
├── configs/mindie_config.json
├── logs/mindie_llm_server.log
├── runtime/
└── pids/mindie_llm_server.pid
~~~

| 位置 | 保存内容 | 使用场景 |
| --- | --- | --- |
| <code>models/</code> | 模型配置、分词器和权重分片 | 查看模型快照 |
| <code>cache/</code> | ModelScope 下载缓存 | 复用已下载文件 |
| <code>configs/</code> | 由模板生成的 MindIE 配置 | 核对模型路径和单卡参数 |
| <code>logs/</code> | 服务输出和基准记录 | 查看启动过程和接口日志 |
| <code>runtime/</code> | MindIE 运行时文件 | 保留服务运行产物 |
| <code>pids/</code> | Notebook 启动的进程号 | 复用或停止服务 |

本实验使用 0 号逻辑 NPU。<code>npu-smi info</code> 用于确认驱动识别到设备；Notebook 随后读取 MindIE 环境脚本、服务命令和配置模板。MindIE 的业务接口监听 <code>127.0.0.1:1025</code>，端侧在 10.02 中通过 SSH 隧道访问该地址。

In [ ]:
DEFAULT_LAB_ROOT = Path('/home/ma-user/work/lab10_end-cloud_dual_model_dialogue_system')
LAB_ROOT = Path(os.environ.get('LAB10_CLOUD_ROOT') or str(DEFAULT_LAB_ROOT)).expanduser().resolve()
MODEL_DIR = LAB_ROOT / 'models' / 'Qwen2-7B-Instruct'
MODELSCOPE_CACHE = LAB_ROOT / 'cache' / 'modelscope'
CONFIG_DIR = LAB_ROOT / 'configs'
CONFIG_FILE = CONFIG_DIR / 'mindie_config.json'
MODEL_SOURCE_FILE = CONFIG_DIR / 'model_source.json'
LOG_DIR = LAB_ROOT / 'logs'
SERVER_LOG = LOG_DIR / 'mindie_llm_server.log'
MINDIE_RUNTIME_DIR = LAB_ROOT / 'runtime'
MINDIE_BENCHMARK_FILE = LOG_DIR / 'mindie_benchmark.jsonl'
PIDS_DIR = LAB_ROOT / 'pids'
PID_FILE = PIDS_DIR / 'mindie_llm_server.pid'

MODEL_ID = 'Qwen/Qwen2-7B-Instruct'
MODEL_REVISION = '39693e977abe9d530648468641ad032a02ba93f7'
SERVED_MODEL_NAME = 'qwen2-7b-instruct'
SERVICE_HOST = '127.0.0.1'
SERVICE_PORT = 1025
SERVICE_URL = f'http://{SERVICE_HOST}:{SERVICE_PORT}'
READINESS_PATH = '/v1/models'
STARTUP_TIMEOUT_SECONDS = 600

ENV_SETUP_SCRIPT = Path('/home/ma-user/mindie-3.0.0-env.sh')
MINDIE_SERVER = ENV_ROOT / 'bin' / 'mindie_llm_server'
MINDIE_TEMPLATE_CONFIG = (
    ENV_ROOT / 'lib' / 'python3.11' / 'site-packages' / 'mindie_llm' / 'conf' / 'config.json'
)

expected_python = ENV_PYTHON.resolve()
actual_python = Path(sys.executable).resolve()
if actual_python != expected_python:
    raise RuntimeError(
        f'当前 Kernel 使用 {actual_python}。请切换到 {KERNEL_DISPLAY_NAME} 后重启并重跑。'
    )

for directory in [LAB_ROOT, MODEL_DIR.parent, MODELSCOPE_CACHE, CONFIG_DIR, LOG_DIR, MINDIE_RUNTIME_DIR, PIDS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
os.chdir(LAB_ROOT)

checks = {
    'MindIE 环境脚本': ENV_SETUP_SCRIPT.is_file(),
    'MindIE 服务命令': MINDIE_SERVER.is_file() and os.access(MINDIE_SERVER, os.X_OK),
    'MindIE 配置模板': MINDIE_TEMPLATE_CONFIG.is_file(),
    '实验目录可写': os.access(LAB_ROOT, os.W_OK),
    'npu-smi 命令': shutil.which('npu-smi') is not None,
}
lab['preflight'] = checks
for name, ok in checks.items():
    print(f"[{'OK' if ok else 'FAIL'}] {name}")

if not all(checks.values()):
    failed = [name for name, ok in checks.items() if not ok]
    raise RuntimeError('运行环境检查未通过：' + '、'.join(failed))

npu_info = subprocess.run(
    ['npu-smi', 'info'], text=True, capture_output=True, check=False, timeout=30
)
if npu_info.returncode != 0:
    raise RuntimeError(npu_info.stderr.strip() or 'npu-smi info 执行失败')
print('\n' + npu_info.stdout)

import modelscope
print('Python：', sys.version.split()[0])
print('ModelScope：', modelscope.__version__)
print('MindIE 服务：', MINDIE_SERVER)
print('服务地址：', SERVICE_URL)

### 3. 下载模型快照

权重来自 ModelScope 的 <code>Qwen/Qwen2-7B-Instruct</code>。模型快照包括权重、<code>config.json</code>、分词器文件和 <code>model.safetensors.index.json</code>；MindIE 加载模型时会读取这些文件。

<code>MODEL_REVISION</code> 固定到一个 Git revision。同一 revision 对应同一份模型内容。Notebook 检查配置、分词器、索引和权重分片，检查通过后复用本地快照。下载结束后，它会记录模型类型、配置摘要和分片大小，并将权限设置为 MindIE 可读取的形式。

华为官方的 [MindIE 3.0 文本生成推理快速入门（以 Qwen2-7B 为例）](https://www.hiascend.com/document/detail/zh/mindie/300/quickstart/docs/zh/user_guide/quick_start.md) 按“准备权重、配置服务、启动、发送请求”的顺序组织流程。本册沿用该顺序，并将运行产物写入实验目录。

In [ ]:
from modelscope import snapshot_download

REQUIRED_MODEL_FILES = ('config.json', 'tokenizer_config.json', 'model.safetensors.index.json')

def model_weight_files(model_dir: Path) -> list[Path]:
    index_path = model_dir / 'model.safetensors.index.json'
    try:
        weight_map = json.loads(index_path.read_text(encoding='utf-8'))['weight_map']
    except (FileNotFoundError, KeyError, TypeError, json.JSONDecodeError):
        return []
    if not isinstance(weight_map, dict) or not weight_map:
        return []
    return [model_dir / name for name in sorted(set(weight_map.values()))]

def model_is_complete(model_dir: Path) -> bool:
    if not model_dir.is_dir() or not all((model_dir / name).is_file() for name in REQUIRED_MODEL_FILES):
        return False
    weight_files = model_weight_files(model_dir)
    return bool(weight_files) and all(path.is_file() and path.stat().st_size > 0 for path in weight_files)

if model_is_complete(MODEL_DIR):
    print('复用已有模型目录：', MODEL_DIR)
else:
    downloaded_path = snapshot_download(
        MODEL_ID,
        revision=MODEL_REVISION,
        cache_dir=str(MODELSCOPE_CACHE),
        local_dir=str(MODEL_DIR),
        max_workers=4,
    )
    print('下载完成：', downloaded_path)

if not model_is_complete(MODEL_DIR):
    raise RuntimeError(f'模型目录不完整：{MODEL_DIR}')

weight_files = model_weight_files(MODEL_DIR)
print('模型目录：', MODEL_DIR)
print('权重分片数：', len(weight_files))
print('权重总大小：%.2f GiB' % (sum(path.stat().st_size for path in weight_files) / 1024**3))

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

model_config_path = MODEL_DIR / 'config.json'
if model_config_path.is_symlink():
    raise RuntimeError('模型 config.json 需要是普通文件，不能是符号链接。')

model_config = json.loads(model_config_path.read_text(encoding='utf-8'))
if model_config.get('model_type') != 'qwen2':
    raise RuntimeError(f"模型类型与预期不符：{model_config.get('model_type')!r}")

for path in sorted(MODEL_DIR.rglob('*')):
    if path.is_symlink():
        raise RuntimeError(f'模型目录包含符号链接：{path}')
    path.chmod(0o750 if path.is_dir() else 0o640)
MODEL_DIR.chmod(0o750)

config_stat = model_config_path.stat()
config_mode = config_stat.st_mode & 0o777
if config_stat.st_uid != os.getuid() or config_stat.st_gid != os.getgid():
    raise RuntimeError(
        '模型 config.json 的属主或属组与当前用户不一致：'
        f'uid={config_stat.st_uid}, gid={config_stat.st_gid}'
    )
if config_mode != 0o640:
    raise RuntimeError(f'模型 config.json 的权限应为 640，实际为 {config_mode:03o}')


model_source = {
    'model_id': MODEL_ID,
    'revision': MODEL_REVISION,
    'local_dir': str(MODEL_DIR),
    'downloaded_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_type': model_config['model_type'],
    'config_sha256': sha256_file(model_config_path),
    'safetensors': [
        {'name': path.name, 'bytes': path.stat().st_size}
        for path in weight_files
    ],
}
MODEL_SOURCE_FILE.write_text(
    json.dumps(model_source, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
MODEL_SOURCE_FILE.chmod(0o640)

print('model_type：', model_source['model_type'])
print('revision：', model_source['revision'])
print('来源记录：', MODEL_SOURCE_FILE)

### 4. 生成 MindIE 配置

本节复制镜像内置模板，并填入模型路径、单卡参数和监听地址。生成后的文件位于 <code>configs/mindie_config.json</code>，服务启动时通过 <code>MIES_CONFIG_JSON_PATH</code> 读取它。

配置可以按三个层次理解：<code>ServerConfig</code> 管理监听地址和端口，<code>BackendConfig</code> 选择 NPU 与模型部署项，<code>ModelConfig</code> 记录模型名、权重路径和卡数。

| 配置字段 | 本实验取值 | 它决定什么 |
| --- | --- | --- |
| <code>ServerConfig.ipAddress</code>、<code>port</code> | <code>127.0.0.1</code>、<code>1025</code> | MindIE 的业务接口位置 |
| <code>httpsEnabled</code> | <code>false</code> | 本实验使用 HTTP 访问本机回环与 SSH 隧道 |
| <code>npuDeviceIds</code> | <code>[[0]]</code> | 选择 0 号逻辑 NPU |
| <code>modelName</code> | <code>qwen2-7b-instruct</code> | 客户端请求中的 <code>model</code> 字段 |
| <code>modelWeightPath</code> | 模型目录绝对路径 | 权重、配置与分词器所在位置 |
| <code>worldSize</code> | <code>1</code> | 一份模型使用一张 NPU |

本实验使用一张逻辑卡，因此配置为 <code>npuDeviceIds=[[0]]</code> 和 <code>worldSize=1</code>。官方 [MindIE 3.0 单机服务部署说明](https://www.hiascend.com/document/detail/zh/mindie/300/mindiemotor/motordev/user_guide/service_deployment/single_machine_service_deployment.md)列出了这些字段在单机部署中的关系。

In [ ]:
shutil.copy2(MINDIE_TEMPLATE_CONFIG, CONFIG_FILE)
mindie_config = json.loads(CONFIG_FILE.read_text(encoding='utf-8'))

server_config = mindie_config['ServerConfig']
backend_config = mindie_config['BackendConfig']
model_configs = backend_config['ModelDeployConfig']['ModelConfig']
if len(model_configs) != 1:
    raise RuntimeError(f'配置模板中的模型实例数为 {len(model_configs)}，需要 1。')

server_config['ipAddress'] = SERVICE_HOST
server_config['port'] = SERVICE_PORT
server_config['httpsEnabled'] = False
backend_config['npuDeviceIds'] = [[0]]

model_config = model_configs[0]
model_config['modelInstanceType'] = 'Standard'
model_config['modelName'] = SERVED_MODEL_NAME
model_config['modelWeightPath'] = str(MODEL_DIR)
model_config['worldSize'] = 1
model_config['trustRemoteCode'] = False

log_config = mindie_config.get('LogConfig')
if isinstance(log_config, dict) and 'logPath' in log_config:
    log_config['logPath'] = str(SERVER_LOG)

CONFIG_FILE.write_text(
    json.dumps(mindie_config, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
CONFIG_DIR.chmod(0o750)
CONFIG_FILE.chmod(0o640)

configured = {
    'ipAddress': server_config['ipAddress'],
    'port': server_config['port'],
    'httpsEnabled': server_config['httpsEnabled'],
    'npuDeviceIds': backend_config['npuDeviceIds'],
    'modelName': model_config['modelName'],
    'modelWeightPath': model_config['modelWeightPath'],
    'worldSize': model_config['worldSize'],
    'trustRemoteCode': model_config['trustRemoteCode'],
}
print(json.dumps(configured, ensure_ascii=False, indent=2))
print('配置文件：', CONFIG_FILE)

### 5. 启动并检查 MindIE 服务

启动单元在后台运行 <code>mindie_llm_server</code>，并将 PID 写入 <code>pids/mindie_llm_server.pid</code>。启动过程依次经过服务进程创建、模型权重加载和接口准备；模型加载阶段通常需要一段等待时间。

Notebook 轮询 <code>/v1/models</code>。该接口返回 <code>qwen2-7b-instruct</code> 后，表示服务已加载模型并可接受后续对话请求。服务日志写入 <code>logs/mindie_llm_server.log</code>。再次执行启动单元时，Notebook 读取 PID 和接口状态，复用已就绪的服务。

In [ ]:
def port_is_open(host: str, port: int, timeout: float = 1.0) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


def health_is_ready(timeout: float = 3.0) -> bool:
    try:
        with urlopen(f'{SERVICE_URL}{READINESS_PATH}', timeout=timeout) as response:
            return response.status == 200
    except (HTTPError, URLError, OSError):
        return False


def read_pid() -> int | None:
    if not PID_FILE.is_file():
        return None
    try:
        return int(PID_FILE.read_text(encoding='utf-8').strip())
    except ValueError:
        PID_FILE.unlink()
        return None


def process_command(pid: int) -> str:
    command_file = Path('/proc') / str(pid) / 'cmdline'
    try:
        return command_file.read_bytes().replace(b'\0', b' ').decode(errors='replace')
    except FileNotFoundError:
        return ''


def pid_is_mindie_server(pid: int) -> bool:
    command = process_command(pid)
    return 'mindie_llm_server' in command or 'mindieservice_daemon' in command


def log_tail(max_chars: int = 5000) -> str:
    if not SERVER_LOG.is_file():
        return '(尚未生成服务日志)'
    return SERVER_LOG.read_text(encoding='utf-8', errors='replace')[-max_chars:]


def signal_mindie(pid: int, sig: signal.Signals) -> None:
    try:
        if os.getpgid(pid) == pid:
            os.killpg(pid, sig)
        else:
            os.kill(pid, sig)
    except ProcessLookupError:
        pass


def wait_for_health(pid: int | None, timeout_seconds: int = STARTUP_TIMEOUT_SECONDS) -> None:
    deadline = time.monotonic() + timeout_seconds
    next_report = time.monotonic()
    while time.monotonic() < deadline:
        if health_is_ready():
            return
        if pid is not None and not pid_is_mindie_server(pid):
            raise RuntimeError('MindIE 进程已退出，日志末尾：\n' + log_tail())
        if time.monotonic() >= next_report:
            remaining = int(deadline - time.monotonic())
            print(f'等待 MindIE 就绪，剩余约 {remaining} 秒。')
            next_report += 10
        time.sleep(2)
    raise TimeoutError('MindIE 未在等待时间内就绪，日志末尾：\n' + log_tail())


def start_mindie() -> int | None:
    existing_pid = read_pid()
    if existing_pid is not None and pid_is_mindie_server(existing_pid):
        print(f'发现本 Notebook 启动的 MindIE，PID={existing_pid}')
        wait_for_health(existing_pid)
        return existing_pid
    if existing_pid is not None:
        PID_FILE.unlink()

    if port_is_open(SERVICE_HOST, SERVICE_PORT):
        if health_is_ready():
            print(f'{SERVICE_URL} 已有健康服务，本次不接管它的生命周期。')
            return None
        raise RuntimeError(
            f'{SERVICE_HOST}:{SERVICE_PORT} 已被占用，且 {READINESS_PATH} 没有就绪响应。'
        )

    command = (
        'source {env}; '
        'unset MIES_CONTAINER_IP MIES_CONTAINER_MANAGEMENT_IP; '
        'unset RANK_TABLE_FILE; '
        'export MINDIE_RUNTIME_PATH={runtime}; '
        'export MINDIE_LLM_BENCHMARK_FILEPATH={benchmark}; '
        'export MINDIE_LOG_TO_STDOUT=1; '
        'export MINDIE_LOG_TO_FILE=0; '
        'export MIES_CONFIG_JSON_PATH={config}; '
        'exec {server}'
    ).format(
        env=shlex.quote(str(ENV_SETUP_SCRIPT)),
        runtime=shlex.quote(str(MINDIE_RUNTIME_DIR)),
        benchmark=shlex.quote(str(MINDIE_BENCHMARK_FILE)),
        config=shlex.quote(str(CONFIG_FILE)),
        server=shlex.quote(str(MINDIE_SERVER)),
    )
    environment = os.environ.copy()
    with SERVER_LOG.open('a', encoding='utf-8') as log_handle:
        process = subprocess.Popen(
            ['bash', '-lc', command],
            cwd=LAB_ROOT,
            env=environment,
            stdout=log_handle,
            stderr=subprocess.STDOUT,
        )
    PID_FILE.write_text(f'{process.pid}\n', encoding='utf-8')
    PID_FILE.chmod(0o640)
    print(f'已启动 MindIE，PID={process.pid}')
    try:
        wait_for_health(process.pid)
    except Exception:
        if process.poll() is not None:
            process.wait()
            PID_FILE.unlink(missing_ok=True)
        raise
    return process.pid

In [ ]:
mindie_pid = start_mindie()
lab['mindie_pid'] = mindie_pid
print('健康检查通过：', f'{SERVICE_URL}{READINESS_PATH}')
print('服务日志：', SERVER_LOG)

### 6. 验证服务接口

MindIE 提供 OpenAI 兼容接口。<code>GET /v1/models</code> 返回已加载的模型列表；<code>POST /v1/chat/completions</code> 接收对话内容并返回生成结果。10.02 中的 Gradio 页面沿用这两个接口请求云端服务。

请求体的 <code>model</code> 字段填写配置中的 <code>modelName</code>。<code>messages</code> 按角色保存系统提示和用户问题，<code>stream</code> 控制响应的返回方式。本单元用一条短消息验证分词、推理和响应解析。

接口字段可对照华为官方的 [MindIE 3.0 OpenAI 兼容接口说明](https://www.hiascend.com/document/detail/zh/mindie/300/mindiellm/llmdev/mindie_service0078.html)。

In [ ]:
def request_json(method: str, path: str, payload: dict | None = None) -> dict | str:
    data = None if payload is None else json.dumps(payload).encode('utf-8')
    request = Request(
        f'{SERVICE_URL}{path}',
        data=data,
        method=method,
        headers={'Content-Type': 'application/json', 'Accept': 'application/json'},
    )
    with urlopen(request, timeout=120) as response:
        body = response.read().decode('utf-8')
    try:
        return json.loads(body)
    except json.JSONDecodeError:
        return body


models = request_json('GET', READINESS_PATH)
print('models：')
print(json.dumps(models, ensure_ascii=False, indent=2))

chat_response = request_json(
    'POST',
    '/v1/chat/completions',
    {
        'model': SERVED_MODEL_NAME,
        'messages': [
            {'role': 'system', 'content': '你是课程实验中的云端助手。回答简洁。'},
            {'role': 'user', 'content': '用一句话说明什么是端云协同推理。'},
        ],
        'temperature': 0.7,
        'max_tokens': 128,
        'stream': False,
    },
)
answer = chat_response['choices'][0]['message']['content']
print('\n模型回答：\n' + answer)

#### 端云访问路径

10.02 将在开发板建立下面的本地转发：

~~~text
开发板 127.0.0.1:1025  ── SSH -L ──►  云端 127.0.0.1:1025
~~~

两个 <code>127.0.0.1:1025</code> 分别属于开发板和云端主机。<code>ssh -L</code> 在开发板创建本地监听端口，请求进入该端口后，经 SSH 连接转发到云端回环地址。MindIE 监听云端回环地址；Gradio 向开发板本机端口发送 HTTP 请求。

### 7. 停止 MindIE 服务

实验结束后执行下面单元。它读取本 Notebook 写入的 PID 文件，核对进程命令后发送结束信号。模型文件、配置文件和日志保留在 Lab 10 目录，可用于下一次实验记录。

In [ ]:
def stop_mindie() -> None:
    pid = read_pid()
    if pid is None:
        print('没有本 Notebook 记录的 MindIE PID。')
        return
    if not pid_is_mindie_server(pid):
        raise RuntimeError(f'PID {pid} 不是 mindie_llm_server，保留 PID 文件供检查。')

    print(f'停止 MindIE，PID={pid}')
    signal_mindie(pid, signal.SIGTERM)
    deadline = time.monotonic() + 30
    while time.monotonic() < deadline and pid_is_mindie_server(pid):
        time.sleep(1)

    if pid_is_mindie_server(pid):
        signal_mindie(pid, signal.SIGKILL)
        time.sleep(1)
    if pid_is_mindie_server(pid):
        raise RuntimeError(f'PID {pid} 仍在运行，请查看 {SERVER_LOG}')

    PID_FILE.unlink(missing_ok=True)
    print('MindIE 已停止。')
    if port_is_open(SERVICE_HOST, SERVICE_PORT):
        print(f'端口 {SERVICE_PORT} 仍可访问，可能有其他服务占用。')


stop_mindie()

## 本节总结

本册完成模型快照下载、MindIE 配置生成和本机接口验证。10.02 会从开发板建立 SSH 隧道，并在 Gradio 中切换端侧与云端的请求函数。

将 <code>/v1/models</code> 返回的模型名与 10.02 请求体中的 <code>model</code> 字段对照；两处均为 <code>qwen2-7b-instruct</code>。